## Currently meant to run locally - adjust path as needed!!!!!

1st normalization then compression.

10k Hz is performing better than 4k¶

In [47]:
import os
#test locally - PoC
# where it's running
current_dir = os.getcwd()
print(f"you are in: {current_dir}")

# folder:
audio_folder = "../raw_data/audio_and_txt_files/" 

if os.path.exists(audio_folder):
    files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
    print(f"{len(files)} files .wav")
else:
    print("not found")

you are in: /home/totid/code/antonella-dm/projeto/notebooks
920 files .wav


In [48]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm #shows progress
import os
import sys
import librosa.util
from scipy.stats import skew, kurtosis

# Features

In [49]:
def extract_features_raw(y, sr):
    
    # Compression - apply Mu-Law compression (standard in audio to enhance low-level signals)
    # 'compress' peaks and amplify breathing details
    mu = 255
    y_compressed = np.sign(y) * np.log1p(mu * np.abs(y)) / np.log1p(mu)
    
    #
    y = y_compressed # y here is the wave sound signl
    
    #end of compression - this can be hidden for testing preprocessing

    
    feat = {}
     # TEMPORAL
    #feat['rms_mean'] = float(np.mean(librosa.feature.rms(y=y)))
    #feat['zcr_mean'] = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))

    # TEMPORAL
    rms = librosa.feature.rms(y=y)
    feat['rms_mean'] = float(np.mean(rms))
    feat['rms_std'] = float(np.std(rms)) # sound explosion

    zcr = librosa.feature.zero_crossing_rate(y=y)
    feat['zcr_mean'] = float(np.mean(zcr))
    
    # SPECTRAL
    #centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    #feat['centroid_mean'] = float(np.mean(centroid))

    # SPECTRAL 
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr) #.flatten() # gemini told me to do it. centroids are 2D and we need it 1D apparently
    feat['centroid_mean'] = float(np.mean(centroid))
    feat['centroid_std'] = float(np.std(centroid)) # tone change

    #SHAPE STATISTICS
    flatness = librosa.feature.spectral_flatness(y=y)
    feat['flatness_mean'] = float(np.mean(flatness))
    feat['flatness_std'] = float(np.std(flatness)) # constant noise vs intermitent


    #OTHERS    
    #feat['centroid_mean'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    feat['rolloff_mean'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    #feat['flatness_mean'] = float(np.mean(librosa.feature.spectral_flatness(y=y))) added it up and split
    feat['flux_mean'] = float(np.mean(librosa.onset.onset_strength(y=y, sr=sr)))
    feat['bandwidth_mean'] = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))) #NEW!!

    # FORM NEW! - depend on centroid
    feat['skewness_mean'] = float(skew(centroid, axis=1)[0])
    feat['kurtosis_mean'] = float(kurtosis(centroid, axis=1)[0])
    
    # MFCC (12, ignore 0) - from 12 increased to 15 or 19
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=16) # 12 mfccs + other 6 = 18 total
    for i in range(1, 16):
        #feat[f'mfcc_{i}'] = float(np.mean(mfccs[i])) # this was before, when not having std and mean
        feat[f'mfcc_{i}_mean'] = float(np.mean(mfccs[i]))
        feat[f'mfcc_{i}_std'] = float(np.std(mfccs[i]))

        
    return feat

# SANITY CHECK

In [50]:
# sanit xheck
print("sanit check")

# white noise
mock_sr = 10000 #instead of the std 22050Hz
mock_y = np.random.uniform(-1, 1, mock_sr * 6)

#
test_feat = extract_features_raw(mock_y, mock_sr)

# chekc
errors = []
if not isinstance(test_feat, dict): errors.append("must be a dict")
if len(test_feat) != 18: errors.append(f" 18 cols ideal, but this one has {len(test_feat)}.")
if any(np.isnan(list(test_feat.values()))): errors.append("NaN in features")

if not errors:
    print("done - 18 vlid columns")
    # example
    print(f"MFCC_1 example: {test_feat.get('mfcc_1'):.4f}")
    print(f"RMS example: {test_feat.get('rms_mean'):.4f}")
else:
    print("errors:")
    for err in errors: print(f" - {err}")

sanit check
errors:
 -  18 cols ideal, but this one has 42.


# MAPS

In [51]:
# raw_data audios - adjust path as needed
diagnosis_path = "../raw_data/patient_diagnosis.csv" 

if os.path.exists(diagnosis_path):
    # adjust names CSV
    df_diag = pd.read_csv(diagnosis_path, names=['patient_id', 'diagnosis'])
    
    # create a dict for ultra-fast lookup: { '101': 'COPD', ... }
    diag_map = dict(zip(df_diag['patient_id'].astype(str), df_diag['diagnosis']))
    print(f"mapping created for {len(diag_map)} patients.")
else:
    print(f"not found")
    diag_map = {}

mapping created for 126 patients.


# new block: DEMOGRAPHICS 
demo_path = "../raw_data/demographic_info.txt"

if os.path.exists(demo_path):
    df_demo = pd.read_csv(demo_path, sep=' ', header=None, 
                          names=['patient_id', 'age', 'sex', 'bmi', 'weight', 'height'])
    
    # dict: { '101': {'age': 60, 'sex': 'M', ...}, ... }
    df_demo['patient_id'] = df_demo['patient_id'].astype(str)
    demo_map = df_demo.set_index('patient_id').to_dict('index')
    print(f"demographics map created for {len(demo_map)} patients.")
else:
    print("demographic file not found!")
    demo_map = {}

### 3.2 Handling Missing Values (Demographics)

* **Age:** NaNs as **median**- to avoid outlier influence.
* **Sex:**
    * `0`: Female
    * `1`: Male
    * `-1`: Unknown/Missing (keep it neutral).

In [53]:
# configuration
TARGET_SR = 12000 # instead of the std 20050 Hz
CHUNK_DURATION = 6 # original was 6s
STEP_DURATION = 2  # it defines the 1/2/3s window (so we can overlap) - can be commented if we want not to have it. ig chunk is 4, then define as 2
all_results = []

## BLACKLIST - UPDATE THIS BASED ON BLACKLIST DRAFT!!
demo_blacklist = ['142', '191', '182'] # replace here

all_results = []
files_skipped = 0


# using all files
print(f"processing {len(files)} files...") # this one can also be out of blacklist block

for f in tqdm(files):
    #extract id to checl blacklist *before* processing
    p_id = f.split('_')[0]
    
    # check if patient is for demo
    if p_id in demo_blacklist:
        files_skipped += 1
        continue # makes it skip file and go to the next
    
    path = os.path.join(audio_folder, f)
    # after this, normal processing

#print(f"done. {files_skipped} files blacklisted.")

## END OF BLACKLIST BLOCK

#for f in tqdm(files):
    #path = os.path.join(audio_folder, f)
    
    try:
        # PREPROCESSING
        
        # Load + Resample + Normalization? check
        # standardizes all audio to Target_SR and scales amplitude to [-1, 1]
        y, sr = librosa.load(path, sr=TARGET_SR)
        
        # Cleaning - trim
        # cutting edges (beginning and end) - not structured as mia did
        y, _ = librosa.effects.trim(y)

        # IDENTIFICATION (extract ID to define slicing)
        p_id = f.split('_')[0]
        diagnosis = diag_map.get(p_id, "Unknown")

#NEW BLOCK OF SLICING WITH OVERLAP - can be commented
        # dynamic slicing
        samples_per_chunk = CHUNK_DURATION * TARGET_SR
        
        # if COPD, no overlap
        if diagnosis == 'COPD':
            step_size = samples_per_chunk 
        else:
            step_size = STEP_DURATION * TARGET_SR
            
        chunks = []
        for start_idx in range(0, len(y) - samples_per_chunk + 1, step_size):
            end_idx = start_idx + samples_per_chunk
            chunk = y[start_idx : end_idx]
            chunks.append(chunk)
#END OF BLOCK WITH SLICING OVERLAP 

        #OR!!!!!!!!!!!!

#SIMPLE 6 s slicing   (can be commented or uncommentend)     
        # slicing (6s windows) -breaks the cleaned audio into fixed 6-second segments
        #samples_per_chunk = CHUNK_DURATION * TARGET_SR 
        #chunks = [y[i : i + samples_per_chunk] for i in range(0, len(y), samples_per_chunk) 
                  #if len(y[i : i + samples_per_chunk]) == samples_per_chunk]
#END of simple 6S SLICING

        # FEATURE EXTRACTION - this cell is getting too long - maybe adjust it later
        
        
        for i, chunk in enumerate(chunks):
            # extract 18 or x features from the 6s processed chunk
            features = extract_features_raw(chunk, TARGET_SR) # it calculates the mean for the 6s slice and return the X features

            # get demo data from patient
            #p_info = demo_map.get(p_id, {})

            # demographic info - CAN be commented
            #features['age'] = p_info.get('age', np.nan)
            #features['bmi'] = p_info.get('bmi', np.nan)

            # sex: 1 for M, 0 for F, -1 if not found - can be commented
            #gender_raw = p_info.get('sex')
            #features['gender'] = 1 if gender_raw == 'M' else (0 if gender_raw == 'F' else -1)
            
            # map diagnosis from our diag_map (fallback to 'unknown')
            features['diagnosis'] = diag_map.get(p_id, "Unknown") # added to labels
            
            # add metadata and labels
            features['patient_id'] = p_id
            features['chunk_id'] = i
            features['original_file'] = f
            #features['diagnosis'] = diagnosis 

            
            all_results.append(features)
            
    except Exception as e:
        print(f"error on{f}: {e}")

# final df
df_final = pd.DataFrame(all_results)

# check
print(f"\ndone - dataset w {len(df_final)} rows.")
print(f"unique diagnoses found: {df_final['diagnosis'].unique()}")
print(f"files skipped for demo: {files_skipped}")

processing 920 files...


100%|█████████████████████████████████████████████████████████████████████████████████| 920/920 [01:53<00:00,  8.10it/s]


done - dataset w 3571 rows.
unique diagnoses found: ['COPD' 'URTI' 'Pneumonia' 'Healthy' 'Bronchiolitis' 'Bronchiectasis'
 'LRTI' 'Asthma']
files skipped for demo: 5


# OPTIONAL - clean NaNs
if not df_final.empty:
    # age: median
    age_median = df_final['age'].median()
    df_final['age'] = df_final['age'].fillna(age_median)
    
    # BMI: median
    bmi_median = df_final['bmi'].median()
    df_final['bmi'] = df_final['bmi'].fillna(bmi_median)
    
    # sex: -1, just to be safe
    df_final['gender'] = df_final['gender'].astype(int)

    print(f"done. age median: {age_median:.1f}")

In [55]:
# check if blacklist were indeed skipped

check_leakage = df_final[df_final['patient_id'].isin(demo_blacklist)]

if check_leakage.empty:
    print("all good! no blacklist file in df.")
    print(f"dataset clean and w {len(df_final)} lines")
else:
    print("AHHHHH! we found the following IDs, that were blacklisted:")
    print(check_leakage['patient_id'].unique())
    print(f"leaked lines: {len(check_leakage)}")

all good! no blacklist file in df.
dataset clean and w 3571 lines


In [42]:
display(df_final.head(15))

,rms_mean,rms_std,zcr_mean,centroid_mean,centroid_std,flatness_mean,flatness_std,rolloff_mean,flux_mean,bandwidth_mean,...,mfcc_13_mean,mfcc_13_std,mfcc_14_mean,mfcc_14_std,mfcc_15_mean,mfcc_15_std,patient_id,chunk_id,original_file,diagnosis
0,0.740132,0.083106,0.001316,98.066230,93.868556,0.000034,0.000146,119.431516,1.826715,340.148743,...,8.284259,2.947182,8.153688,3.566122,7.156815,3.025179,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.696974,0.079933,0.001572,94.479942,81.417895,0.000041,0.000361,107.338763,1.494651,338.655438,...,8.452517,3.363525,7.557367,4.090311,6.353885,4.028513,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.670958,0.079359,0.001590,90.514388,75.801911,0.000024,0.000205,99.817154,1.613105,339.537651,...,8.950355,2.893862,7.780566,3.040372,6.503709,3.147859,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.675636,0.093185,0.001153,81.297469,52.764923,0.000008,0.000023,81.740359,1.693879,322.028853,...,8.722843,2.706289,8.697692,3.737317,6.902027,3.432181,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.635533,0.069224,0.001202,80.768717,67.285689,0.000014,0.000083,81.865027,1.705645,325.092538,...,9.458111,3.485257,10.628664,4.948114,8.951859,4.647463,223,4,223_1b1_Pr_sc_Meditron.wav,COPD
5,0.572379,0.045065,0.002421,132.623717,53.403051,0.000034,0.000165,152.967088,1.019266,445.912796,...,6.503563,2.083931,5.543195,2.200125,5.548570,1.839354,134,0,134_2b2_Al_mc_LittC2SE.wav,COPD
6,0.564768,0.055911,0.002729,139.824071,64.152380,0.000046,0.000265,159.990027,1.026882,469.288731,...,6.878557,2.196946,5.460019,1.750543,5.138194,1.740964,134,1,134_2b2_Al_mc_LittC2SE.wav,COPD
7,0.576551,0.053092,0.002580,133.112183,54.665975,0.000028,0.000120,152.759309,0.972355,446.442633,...,6.872697,1.862711,5.341773,1.597097,5.164229,1.627355,134,2,134_2b2_Al_mc_LittC2SE.wav,COPD
8,0.561548,0.100226,0.033851,944.255032,359.733133,0.013931,0.022490,2253.781582,1.789640,1362.351319,...,4.400384,4.217227,1.971999,4.555805,3.420474,3.220721,170,0,170_1b4_Al_mc_AKGC417L.wav,COPD
9,0.546012,0.107101,0.024916,842.735606,244.973042,0.006972,0.006661,1994.099069,1.704101,1300.777019,...,5.813795,3.688287,2.093760,3.865756,5.509666,2.887885,170,1,170_1b4_Al_mc_AKGC417L.wav,COPD


In [43]:
print(df_final.columns.tolist())

['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std', 'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean', 'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std', 'mfcc_15_mean', 'mfcc_15_std', 'patient_id', 'chunk_id', 'original_file', 'diagnosis']


In [44]:
print(df_final['diagnosis'].value_counts().head(8))

diagnosis
COPD              2597
Pneumonia          296
Healthy            275
URTI               182
Bronchiectasis     128
Bronchiolitis      104
LRTI                16
Asthma               8
Name: count, dtype: int64


In [45]:
# Save CSV to raw_data
df_final.to_csv("../raw_data/xgboost_features_6s_12kHz_compressed_after_normalization_w_overlapping_mfcc15_no_demographics.csv", index=False)
print("done")

done
